<a href="https://colab.research.google.com/github/starbot2001/CS336-Assignemnt_1/blob/main/CS336_Lec_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/stanford-cs336/assignment1-basics.git
import os
os.chdir('assignment1-basics')

# Install dependencies and the package in editable mode
!pip install -e .[test]
!pip install uv
!uv run pytest
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
!gunzip owt_train.txt.gz
!wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
!gunzip owt_valid.txt.gz

!cd ..

Cloning into 'assignment1-basics'...
remote: Enumerating objects: 263, done.
remote: Total 263 (delta 0), reused 0 (delta 0), pack-reused 263 (from 1)
Receiving objects: 100% (263/263), 14.68 MiB | 21.69 MiB/s, done.
Resolving deltas: 100% (112/112), done.
Obtaining file:///content/assignment1-basics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 21.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 110.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 100.1 MB/s  0:00:00
  Building editable for cs336_basics (pyproject.toml) ... done
  Created wheel for cs336_basics: filename=cs336_basics-26.0.0-py3-none-any.whl size=2026 sha256=7ca1ed79eb09bcdaf978f2df3772fb21a26dc8979cee4f92325aa22afdb20035
  Stored in directory: /tmp/pip-e

 # Byte-Pair Encoding (BPE) Tokenizer

In [ ]:
#2.1

chr(29275),ord("牛")

('牛', 29275)

In [ ]:
print(chr(29275)),chr(29275)

牛


(None, '牛')

In [ ]:
print("this is a test" + chr(0) + "string")

this is a test string


In [ ]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [ ]:
test_string = "hello! こんにちは!"

In [ ]:
utf_encoded=test_string.encode("utf-8")

In [ ]:
print(utf_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [ ]:
len(list(utf_encoded))

23

In [ ]:
len(list(test_string))

13

In [ ]:
utf_encoded.decode("utf-8")

'hello! こんにちは!'

In [ ]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
  return "".join([bytes([b]).decode("utf-8") for b in bytestring])

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
  return bytestring.decode("utf-8")



In [ ]:
decode_utf8_bytes_to_str_wrong( "🚀" .encode("utf-8"))

'🚀'

In [16]:
!head -n 20 /content/assignment1-basics/TinyStoriesV2-GPT4-train.txt


Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  
He said, “Wow, that is a really amazing vase! Can I buy it?” 
The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”
So Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. 
And that's how Ben found an amazing vase in the store!
<|endoftext|>
Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
One day, Ollie's mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He saw his fri

In [22]:
import os
from typing import BinaryIO
from multiprocessing import Pool
from collections import Counter
import regex as re

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    return sorted(set(chunk_boundaries))

file_path_train="/content/assignment1-basics/TinyStoriesV2-GPT4-train.txt"
# GPT-4 Pre-tokenization regex pattern
PAT = r"""<\|endoftext\|>|'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
num_processes = 4

# Optimized chunk processor
def chunk_processor(file_path, start, end):
    counts = Counter()
    with open(file_path, 'rb') as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        tokens = re.findall(PAT, chunk)

        # Filter out '<|endoftext|>' from the counts so it is excluded from BPE merge training
        filtered_tokens = [t for t in tokens if t != "<|endoftext|>"]
        counts.update(filtered_tokens)
    return counts

## Usage
if __name__ == '__main__':
    # 1. Quick Sample Run / Sanity Check
    print("--- Running a quick single-sample dry run (first 10,000 bytes) ---")
    with open(file_path_train, "rb") as f:
        sample_chunk = f.read(10000).decode("utf-8", errors="ignore")
        sample_tokens = re.findall(PAT, sample_chunk)
        # Demonstrate excluding the special token during vocabulary count
        sample_filtered = [t for t in sample_tokens if t != "<|endoftext|>"]
        sample_counts = Counter(sample_filtered)
        print("Sample run completed successfully!")
        print("Sample unique tokens found (excluding special tokens):", len(sample_counts))
        print("Sample most common tokens:", sample_counts.most_common(5))
        print("Count of '<|endoftext|>' (should be 0):", sample_counts.get('<|endoftext|>', 0))
        print("-----------------------------------------------------------\n")

    # 2. Full Parallel Execution without loop overhead
    with open(file_path_train, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
        print("Chunk boundaries:", boundaries)
        exec_tup = [(file_path_train, start, end) for start, end in zip(boundaries[:-1], boundaries[1:])]
        print("Tasks assigned:", exec_tup)

    print("Starting parallel pre-tokenization...")
    with Pool(processes=num_processes) as pool:
        results = pool.starmap(chunk_processor, exec_tup)

        # Combine the counts from all processes
        global_counts = Counter()
        for r in results:
            global_counts.update(r)

        print(f"Successfully processed {len(results)} chunks.")
        print("Unique pre-tokens found:", len(global_counts))
        print("Top 10 most common pre-tokens:", global_counts.most_common(10))

--- Running a quick single-sample dry run (first 10,000 bytes) ---
Sample run completed successfully!
Sample unique tokens found (excluding special tokens): 523
Sample most common tokens: [('.', 188), (',', 97), (' the', 91), (' and', 91), ('\n', 70)]
Count of '<|endoftext|>' (should be 0): 0
-----------------------------------------------------------

Chunk boundaries: [0, 556938507, 1113876594, 1670815354, 2227753162]
Tasks assigned: [('/content/assignment1-basics/TinyStoriesV2-GPT4-train.txt', 0, 556938507), ('/content/assignment1-basics/TinyStoriesV2-GPT4-train.txt', 556938507, 1113876594), ('/content/assignment1-basics/TinyStoriesV2-GPT4-train.txt', 1113876594, 1670815354), ('/content/assignment1-basics/TinyStoriesV2-GPT4-train.txt', 1670815354, 2227753162)]
Starting parallel pre-tokenization...


KeyboardInterrupt: 